In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("MERGED.csv")

df = df[df['WHO_Region'] != 'EURO']
df['Population'] = df['pop_t'] * 1000
df['Incidence'] = df['Autochthonous_Cases'] / df['Population'] * 100000

df = df.sort_values(['Country', 'Year'])
df['Incidence_next'] = df.groupby('Country')['Incidence'].shift(-1)
df = df.dropna(subset=['Incidence_next'])

print("Shape after preprocessing:", df.shape)

Shape after preprocessing: (985, 132)


In [39]:
ladder_groups = {}
for area in ['r', 'u', 't']:
    ladder_groups[f'wat_{area}'] = [f'wat_basal_{area}', f'wat_lim_{area}', f'wat_unimp_{area}', f'wat_ns_{area}']
    ladder_groups[f'san_{area}'] = [f'san_basal_{area}', f'san_lim_{area}', f'san_unimp_{area}', f'san_ns_{area}']
    ladder_groups[f'hyg_{area}'] = [f'hyg_bas_{area}',   f'hyg_lim_{area}',   f'hyg_ns_{area}']

for group_name, cols in ladder_groups.items():
    row_sums = df[cols].sum(axis=1, skipna=False)
    print(f"{group_name}: mean={row_sums.mean():.2f}  min={row_sums.min():.2f}  "
          f"max={row_sums.max():.2f}  n_valid={row_sums.notna().sum()}")

wat_r: mean=100.00  min=100.00  max=100.00  n_valid=900
san_r: mean=100.00  min=100.00  max=100.00  n_valid=886
hyg_r: mean=100.00  min=100.00  max=100.00  n_valid=479
wat_u: mean=100.00  min=100.00  max=100.00  n_valid=934
san_u: mean=100.00  min=100.00  max=100.00  n_valid=915
hyg_u: mean=100.00  min=100.00  max=100.00  n_valid=476
wat_t: mean=100.00  min=100.00  max=100.00  n_valid=963
san_t: mean=100.00  min=100.00  max=100.00  n_valid=969
hyg_t: mean=100.00  min=100.00  max=100.00  n_valid=482


In [40]:
def clr_transform(sub_df, epsilon=1e-6):
    props = sub_df.div(sub_df.sum(axis=1), axis=0)
    props = props.clip(lower=epsilon)
    props = props.div(props.sum(axis=1), axis=0)
    log_props = np.log(props)
    return log_props.sub(log_props.mean(axis=1), axis=0)

id_target_cols = ['Country', 'Year', 'WHO_Region', 'Autochthonous_Cases',
                   'Population', 'Incidence', 'Incidence_next', 'pop_t', 'prop_u']

ladder_cols_flat = [c for cols in ladder_groups.values() for c in cols]

other_feature_cols = [c for c in df.columns if c not in id_target_cols + ladder_cols_flat]

df_normalized = df.copy()
scale_cols = ladder_cols_flat + [c for c in other_feature_cols if df[c].dtype != 'object']
df_normalized[scale_cols] = df_normalized[scale_cols] / 100

df_coda = df.copy()
for group_name, cols in ladder_groups.items():
    clr_vals = clr_transform(df[cols])
    clr_vals.columns = [f'{c}_clr' for c in cols]
    df_coda = df_coda.drop(columns=cols)
    df_coda = pd.concat([df_coda, clr_vals], axis=1)

df_normalized.to_csv("MERGED_normalized.csv", index=False)
df_coda.to_csv("MERGED_coda.csv", index=False)

print(df_normalized.shape, df_coda.shape)

(985, 132) (985, 132)
